# Serp Hybrid Product URL Finder

2 Google organic searches + up to 2 Google AI Mode validations per product.


In [12]:
import sys
import os
from pathlib import Path
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
ENV_PATH = PROJECT_ROOT / ".env"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

load_dotenv(ENV_PATH, override=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("SRC_PATH:", SRC_PATH)
print("ENV_PATH:", ENV_PATH)
print(".env exists:", ENV_PATH.exists())

PROJECT_ROOT: /mnt/batch/tasks/shared/LS_root/mounts/clusters/akilankm/code/Users/Akilan.Km/TechDurablesToy/web_search_tool
SRC_PATH: /mnt/batch/tasks/shared/LS_root/mounts/clusters/akilankm/code/Users/Akilan.Km/TechDurablesToy/web_search_tool/src
ENV_PATH: /mnt/batch/tasks/shared/LS_root/mounts/clusters/akilankm/code/Users/Akilan.Km/TechDurablesToy/web_search_tool/.env
.env exists: True


In [ ]:
from serp_hybrid_url_finder import (
    CSVProductIO,
    HybridProductURLFinderPipeline,
    PipelineConfig,
    ProductQuery,
    RichPrinter,
    SerpAPIConfig,
    configure_logging,
)
import asyncio
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm
from datetime import datetime

configure_logging('INFO')
printer = RichPrinter()

In [14]:
serp_config = SerpAPIConfig.from_env(
    country_code='CO',  # SerpAPI gl fallback; per-product country_code overrides this
    language_code='en',
    no_cache=False,
)

pipeline_config = PipelineConfig(
    max_organic_calls=2,
    max_ai_mode_calls=2,
    max_candidates_for_ai=18,
    run_ai_repair=True,
    repair_confidence_threshold=0.80,
    # crawl4ai scrape verification
    scrape_enabled=True,
    require_scrapable_final=True,
    max_urls_to_scrape=10,
    crawl_headless=True,
)

pipeline = HybridProductURLFinderPipeline(
    serp_config=serp_config,
    pipeline_config=pipeline_config,
)


## Single product run


In [15]:
product = ProductQuery(
    row_id='demo-001',
    main_text='MATCHBOX LESNEY MADE IN ENGLAND #7  - NARANJA',
    country_code='CO',        # required
    retailer_name='meli',       # optional
    ean='8018190039368',      # optional
)

trace = pipeline.run(product, return_trace=True)


2026-06-20 15:46:11 | INFO     | serp_hybrid_url_finder.pipeline:run:100 - Starting hybrid product URL finding | row_id=demo-001
2026-06-20 15:46:11 | INFO     | serp_hybrid_url_finder.serp_clients:search:192 - Calling Google organic search | query_chars=84 | query="MATCHBOX LESNEY MADE IN ENGLAND #7 - NARANJA" 8018190039368 meli CO product produkt
2026-06-20 15:46:13 | INFO     | serp_hybrid_url_finder.serp_clients:search:238 - Organic search returned 9 result(s)
2026-06-20 15:46:13 | INFO     | serp_hybrid_url_finder.serp_clients:search:192 - Calling Google organic search | query_chars=189 | query=8018190039368 "MATCHBOX LESNEY MADE IN ENGLAND #7 - NARANJA" meli CO product detail official product page retailer product page -youtube -facebook -instagram -tiktok -pinterest -pdf -manual
2026-06-20 15:46:19 | WARNING  | serp_hybrid_url_finder.serp_clients:search:202 - Organic search returned no results | query=8018190039368 "MATCHBOX LESNEY MADE IN ENGLAND #7 - NARANJA" meli CO product d

In [16]:
printer.print_budget(trace.budget)

╭──────────────────────────────────────────── Per-Product Call Budget ────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━┓                                                                                   │
│ ┃ Metric            ┃ Value ┃                                                                                   │
│ ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━┩                                                                                   │
│ │ organic_used      │ 2     │                                                                                   │
│ │ organic_remaining │ 0     │                                                                                   │
│ │ max_organic       │ 2     │                                                                                   │
│ │ ai_mode_used      │ 2     │                                                                                   │
│ │ ai_mode_remaining │ 0     │                                                                                   │
│ │ max_ai_mode       │ 2     │                                                                                   │
│ └───────────────────┴───────┘                                                                                   │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [17]:

printer.print_match(trace.best_match)

╭──────────────────────────────────────────── Best Product URL Match ─────────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│ ┃ Field                  ┃ Value                                                                              ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ │
│ │ row_id                 │ demo-001                                                                           │ │
│ │ main_text              │ MATCHBOX LESNEY MADE IN ENGLAND #7  - NARANJA                                      │ │
│ │ ean                    │ 8018190039368                                                                      │ │
│ │ retailer_name          │ meli                                                                               │ │
│ │ country_code           │ CO                                                                                 │ │
│ │ product_url            │ https://articulo.mercadolibre.com.co/MCO-2332455320-matchbox-lesney-1969-made-in-e │ │
│ │                        │ ngland-_JM                                                                         │ │
│ │ confidence             │ 0.55                                                                               │ │
│ │ validation_status      │ NEEDS_REVIEW                                                                       │ │
│ │ identity_status        │ PROBABLE                                                                           │ │
│ │ is_exact_product_match │ False                                                                              │ │
│ │ justification          │ Identity: PROBABLE. Evidence: distinctive title tokens matched (4/5): matchbox,    │ │
│ │                        │ lesney, made, england; brand token present on page. Submission status:             │ │
│ │                        │ NEEDS_REVIEW.                                                                      │ │
│ │ match_reason           │ identity=PROBABLE; ean_check=ABSENT; qty_check=NOT_APPLICABLE; title_check=STRONG; │ │
│ │                        │ page_type=UNKNOWN; retailer/domain matched; country signal matched; verified       │ │
│ │                        │ scrapable via crawl4ai; scrape_status=302; ai_decision=NO_MATCH                    │ │
│ │ ean_check              │ ABSENT                                                                             │ │
│ │ title_check            │ STRONG                                                                             │ │
│ │ quantity_check         │ NOT_APPLICABLE                                                                     │ │
│ │ page_type_check        │ UNKNOWN                                                                            │ │
│ │ retailer_check         │ MATCHED                                                                            │ │
│ │ country_check          │ MATCHED                                                                            │ │
│ │ requested_quantity     │                                                                                    │ │
│ │ page_quantity          │                                                                                    │ │
│ │ blocking_reasons       │                                                                                    │ │
│ │ ai_match_decision      │ NO_MATCH                                                                           │ │
│ │ ai_confidence_reason   │                                                                                    │ │
│ │ ean_evidence           │ not_provided                                                                       │ │
│ │ title_evidence         │ weak                                                                               │ │
│ │ retailer_evidence      │ not_provided               

In [18]:

printer.print_candidates(trace.scored_candidates, max_rows=15)

                                                 Ranked Candidates                                                 
┏━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━┓
┃ Rank ┃ Confidence ┃ Validati… ┃  Identity  ┃  EAN   ┃    Qty    ┃ Retailer ┃ Scrapable ┃ URL        ┃ Reason    ┃
┡━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━┩
│    1 │      0.550 │ NEEDS_RE… │  PROBABLE  │ ABSENT │ NOT_APPL… │ MATCHED  │    yes    │ https://ar │ identity= │
│      │            │           │            │        │           │          │           │ ticulo.mer │ PROBABLE; │
│      │            │           │            │        │           │          │           │ cadolibre. │           │
│      │            │           │            │        │           │          │           │ com.co/MCO │ ean_check │
│      │            │           │            │        │           │          │           │ -233245532 │ =ABSENT;  │
│      │            │           │            │        │           │          │           │ 0-matchbox │ qty_check │
│      │            │           │            │        │           │          │           │ -lesney-19 │ =NOT_APPL │
│      │            │           │            │        │           │          │           │ 69-made-in │ ICABLE;   │
│      │            │           │            │        │           │          │           │ -england-_ │ title_che │
│      │            │           │            │        │           │          │           │ JM         │ ck=STRONG │
│      │            │           │            │        │           │          │           │            │ ;         │
│      │            │           │            │        │           │          │           │            │ page_type │
│      │            │           │            │        │           │          │           │            │ =UNKNOWN; │
│      │            │           │            │        │           │          │           │            │           │
│      │            │           │            │        │           │          │           │            │ retailer/ │
│      │            │           │            │        │           │          │           │            │ domain    │
│      │            │           │            │        │           │          │           │            │ matched;  │
│      │            │           │            │        │           │          │           │            │ country   │
│      │            │           │            │        │           │          │           │            │ signal    │
│      │            │           │            │        │           │          │           │            │ matched;  │
│      │            │           │            │        │           │          │           │            │ verified  │
│      │            │           │            │        │           │          │           │            │ scrapable │
│      │            │           │            │        │           │          │           │            │ via       │
│      │            │           │            │        │           │          │           │            │ crawl4ai; │
│      │            │           │            │        │           │          │           │            │ scrape_st │
│      │            │           │            │        │           │          │           │            │ atus=302; │
│      │            │           │            │        │           │          │           │            │           │
│      │            │           │            │        │           │          │           │            │ ai_decisi │
│      │            │           │            │        │           │          │           │            │ on=NO_MAT │
│      │            │           │            │        │           │          │           │            │ CH        │
│    2 │      0.500 │ NEEDS_RE… │    WEAK    │ ABSENT │ 

In [9]:

trace.best_match.to_dict()


{'row_id': 'demo-001',
 'main_text': 'PKM ME04 WACHSENDES CHAOS BOOSTER',
 'ean': '0196214141070',
 'retailer_name': None,
 'country_code': 'CH',
 'product_url': 'https://www.tutti.ch/de/vi/basel/sammeln/sammeln/pokemon-wachsendes-chaos-booster-box-display-deutsch-me04/81366253?srsltid=AfmBOoqcCX906YKo_tm9hEZFop-G7W6SA8h281uzYzCHq_Oue4-vkld-',
 'confidence': 0.74,
 'validation_status': 'NEEDS_REVIEW',
 'identity_status': 'PROBABLE',
 'is_exact_product_match': False,
 'justification': 'Identity: PROBABLE. Evidence: distinctive title tokens matched (4/5): me04, wachsendes, chaos, booster. Submission status: NEEDS_REVIEW.',
 'match_reason': 'identity=PROBABLE; ean_check=ABSENT; qty_check=NOT_APPLICABLE; title_check=STRONG; page_type=UNKNOWN; country signal matched; verified scrapable via crawl4ai; scrape_status=307; ai_decision=NO_MATCH',
 'ean_check': 'ABSENT',
 'title_check': 'STRONG',
 'quantity_check': 'NOT_APPLICABLE',
 'page_type_check': 'UNKNOWN',
 'retailer_check': 'NOT_PROVIDED',

## Inspect prompts and AI evidence


In [19]:
print('Organic queries:')
for q in trace.organic_queries:
    print('-', q)

print('\nAI validation query:')
print(trace.ai_validation_query)

print('\nAI validation markdown:')
print(trace.ai_validation_response.markdown)

if trace.repair_response:
    print('\nAI repair query:')
    print(trace.repair_query)
    print('\nAI repair markdown:')
    print(trace.repair_response.markdown)


Organic queries:
- "MATCHBOX LESNEY MADE IN ENGLAND #7 - NARANJA" 8018190039368 meli CO product produkt
- 8018190039368 "MATCHBOX LESNEY MADE IN ENGLAND #7 - NARANJA" meli CO product detail official product page retailer product page -youtube -facebook -instagram -tiktok -pinterest -pdf -manual

AI validation query:
You are validating product URL candidates using indexed web evidence.
TASK:
Choose the single best exact product detail URL from the candidate list. Use evidence, not guesses. Do not invent a URL. Do not return a URL outside the candidates unless it appears in your cited references.
PRODUCT INPUT:
main_text: "MATCHBOX LESNEY MADE IN ENGLAND #7  - NARANJA"
ean_gtin: "8018190039368"
retailer_name: meli
country_code: CO
VALIDATION RULES:
- Prefer EAN/GTIN/barcode match over fuzzy title match.
- If retailer_name is provided, prefer that retailer's product detail page.
- The final URL must be a single product detail page.
- Reject homepage, brand homepage, category, search, list

In [20]:
print('Scrape + identity verification per candidate:')
for url, scrape in trace.scrapes.items():
    v = trace.verifications.get(url)
    identity = v.identity_status if v else 'NONE'
    ean = v.ean_check if v else '-'
    qty = v.quantity_check if v else '-'
    print(
        f"- identity={identity:10} ean={ean:9} qty={qty:9} "
        f"scrapable={scrape.is_scrapable} soft404={scrape.is_soft_404} url={url}"
    )

print('\nFinal submission status:', trace.best_match.validation_status)
print('Identity status        :', trace.best_match.identity_status)
print('Retailer match         :', trace.best_match.retailer_check,
      '(requested:', trace.best_match.retailer_name, ')')
print('Confidence             :', trace.best_match.confidence)
print('Product URL            :', trace.best_match.product_url)
print('Justification          :', trace.best_match.justification)

# Full identity + confidence breakdown for the top-ranked candidate (auditable).
if trace.scored_candidates:
    printer.print_verification(trace.scored_candidates[0])


Scrape + identity verification per candidate:
- identity=PROBABLE   ean=ABSENT    qty=NOT_APPLICABLE scrapable=True soft404=False url=https://articulo.mercadolibre.com.co/MCO-2332455320-matchbox-lesney-1969-made-in-england-_JM
- identity=WEAK       ean=ABSENT    qty=NOT_APPLICABLE scrapable=True soft404=False url=https://hn.ebay.com/b/Matchbox-1-75-Lesney-Orange-Vintage-Manufacture-Diecast-Cars-Trucks-Vans/180507/bn_111975513
- identity=WEAK       ean=ABSENT    qty=NOT_APPLICABLE scrapable=True soft404=False url=https://www.mercadolibre.com.mx/7-matchbox-lesney-england-vintage-lote-antiguo/up/MLMU3127112440
- identity=WEAK       ean=ABSENT    qty=NOT_APPLICABLE scrapable=True soft404=False url=https://tigrisantiques.com/es/products/leksaker-hobby/matchbox-lesney-modeller-av-yesteryear-england
- identity=UNVERIFIED ean=ABSENT    qty=NOT_APPLICABLE scrapable=True soft404=False url=https://www.ebay.com/b/Matchbox-Lesney/bn_21830275
- identity=UNVERIFIED ean=ABSENT    qty=NOT_APPLICABLE sc

╭───────────────────────────────────────── Product Identity Verification ─────────────────────────────────────────╮
│ ┏━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓ │
│ ┃ Identity Check     ┃ Result                                                                                 ┃ │
│ ┡━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩ │
│ │ url                │ https://articulo.mercadolibre.com.co/MCO-2332455320-matchbox-lesney-1969-made-in-engla │ │
│ │                    │ nd-_JM                                                                                 │ │
│ │ identity_status    │ PROBABLE                                                                               │ │
│ │ ean_check          │ ABSENT                                                                                 │ │
│ │ title_check        │ STRONG                                                                                 │ │
│ │ quantity_check     │ NOT_APPLICABLE                                                                         │ │
│ │ brand_check        │ MATCHED                                                                                │ │
│ │ page_type_check    │ UNKNOWN                                                                                │ │
│ │ title_match_score  │ 0.8                                                                                    │ │
│ │ requested_quantity │                                                                                        │ │
│ │ page_quantity      │                                                                                        │ │
│ │ requested_ean      │ 8018190039368                                                                          │ │
│ │ page_eans          │ []                                                                                     │ │
│ │ matched_tokens     │ ['matchbox', 'lesney', 'made', 'england']                                              │ │
│ │ missing_tokens     │ ['naranja']                                                                            │ │
│ │ justifications     │ ['distinctive title tokens matched (4/5): matchbox, lesney, made, england', 'brand     │ │
│ │                    │ token present on page']                                                                │ │
│ │ blocking_reasons   │ []                                                                                     │ │
│ └────────────────────┴────────────────────────────────────────────────────────────────────────────────────────┘ │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

                                          Confidence Breakdown                                           
┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Component             ┃   Raw ┃ Weight ┃ Contribution ┃ Justification                                 ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ identity_verification │ 0.750 │  0.300 │        0.225 │ Product-identity verdict from scraped content │
│ main_text             │ 1.000 │  0.120 │        0.120 │ Distinctive title-token overlap               │
│ retailer              │ 1.000 │  0.120 │        0.120 │ Retailer match                                │
│ organic_consensus     │ 0.900 │  0.100 │        0.090 │ Organic search corroboration                  │
│ scrape_verification   │ 0.600 │  0.100 │        0.060 │ crawl4ai scrape verdict                       │
│ country               │ 1.000 │  0.060 │        0.060 │ Country signal                                │
│ ai_evidence           │ 0.299 │  0.160 │        0.048 │ AI Mode validation evidence                   │
│ product_page_shape    │ 0.350 │  0.100 │        0.035 │ Product-detail page shape                     │
│ ean                   │ 0.000 │  0.140 │        0.000 │ EAN/GTIN evidence                             │
└───────────────────────┴───────┴────────┴──────────────┴───────────────────────────────────────────────┘

                      Caps Applied                       
┏━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃  Cap ┃ Reason                                         ┃
┡━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ 0.74 │ probable match (not EAN-proven) — needs review │
│ 0.55 │ URL shape is not product-detail-like           │
└──────┴────────────────────────────────────────────────┘

╭────────────────────────────────────────────── Confidence Verdict ───────────────────────────────────────────────╮
│ base=0.858 -> final=0.550 | status=NEEDS_REVIEW                                                                 │
│ Identity: PROBABLE. Evidence: distinctive title tokens matched (4/5): matchbox, lesney, made, england; brand    │
│ token present on page. Submission status: NEEDS_REVIEW.                                                         │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Batch CSV run


In [ ]:
def run_batch_async(products, max_concurrent=5, output_csv=None):
    """Run products concurrently with bounded parallelism, progress, and error handling."""
    matches = []
    errors = []

    def safe_run(product):
        try:
            return pipeline.run(product), None
        except Exception as e:
            return None, (product.row_id, str(e))

    with ThreadPoolExecutor(max_workers=max_concurrent) as executor:
        with tqdm(total=len(products), desc="Processing") as pbar:
            futures = [executor.submit(safe_run, p) for p in products]
            for i, future in enumerate(as_completed(futures)):
                match, error = future.result()
                if error:
                    errors.append(error)
                else:
                    matches.append(match)
                pbar.update(1)
                if output_csv and i % 50 == 0 and matches:
                    CSVProductIO.write_matches(output_csv, matches)

    return matches, errors

# Run batch
products = CSVProductIO.read_products(PROJECT_ROOT / 'data' / 'sample_products.csv')
output_path = PROJECT_ROOT / 'outputs' / 'product_url_mapping.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)

print(f"Starting batch processing of {len(products)} products...")
start = datetime.now()
matches, errors = run_batch_async(products, max_concurrent=5, output_csv=output_path)
elapsed = (datetime.now() - start).total_seconds()

print(f"\n✓ Processed {len(matches)} successfully in {elapsed:.1f}s ({len(matches)/elapsed:.1f} products/sec)")
print(f"✗ Failed: {len(errors)}")
if errors:
    for row_id, err in errors[:5]:
        print(f"  - {row_id}: {err[:80]}")

CSVProductIO.write_matches(output_path, matches)
print(f"✓ Output: {output_path}")

## Batch Configuration Guide

**For 500 products:**
- `max_concurrent=5` → 5 parallel products (adjust based on CPU/SerpAPI rate limits)
- Each product = ~2 SerpAPI organic + ~1 AI Mode call + crawl4ai scrape
- Estimated time: 500 ÷ 5 × (avg 45s per product) ≈ 75 min

**Tuning:**
- Increase `max_concurrent` for faster runs (but monitor SerpAPI quota)
- Decrease if hitting timeout/rate limit errors
- Results written every 50 products (resumable via append)

**Data files needed:**
- `data/sample_products.csv` with columns: `row_id`, `main_text`, `country_code`, `retailer_name` (optional), `ean` (optional)
- Output: `outputs/product_url_mapping.csv` with full match data + confidence breakdown

In [ ]:
# Quick check: existing sample products
sample = CSVProductIO.read_products(PROJECT_ROOT / 'data' / 'sample_products.csv') if (PROJECT_ROOT / 'data' / 'sample_products.csv').exists() else []
print(f"Found {len(sample)} products in data/sample_products.csv")
if sample:
    print(f"Sample: {sample[0]}")
else:
    print("⚠ Create data/sample_products.csv first (see Batch Configuration Guide above)")